# Generative AI Temelleri 

## 1. GenAI Temelleri ve Kavramlar

* **LLM:** İstemleri olasılıksal olarak sürdüren, milyarlarca parametreli ve geniş metinlerle eğitilmiş genel amaçlı dil modelidir.
* **Transformer Mimarisi:** Girdideki öğeler arası bağıntıları self-attention ile modelleyen, paralel hesaplamaya elverişli derin sinir ağı mimarisidir.
* **Token:** Modelin işlediği en küçük birimdir; kelime, alt kelime veya karakter olabilir ve uzunluk/maliyet bu birimlerle ölçülür.
* **Context Window:** Modelin tek seferde işleyip “hatırlayabildiği” azami token bütçesidir (girdi+çıktı) ve aşıldığında önceki bağlam dışarıda kalır.
* **System Instructions:** Modelin rolünü, sınırlarını ve üslubunu belirleyen, diğer iletilere göre önceliği yüksek üst seviye talimatlardır.
* **Hallucination:** Modelin gerçek veriye dayanmayan fakat tutarlı görünen bilgi uydurmasıdır ve RAG/ek doğrulama ile azaltılabilir.
* **Thinking (Düşünme):** Modelin cevap vermeden önce bir iç muhakeme süreci yürütmesidir. Gemini 3 ve 2.5 serisi modellerde varsayılan olarak açıktır. `thinking_budget` parametresi ile kontrol edilir: 0 = kapalı, -1 = otomatik.
* **Multimodal:** Tek bir modelin metin, görsel, ses ve video gibi farklı veri tiplerini aynı anda anlayıp işleyebilmesidir.

## 2. LLM-Based Uygulama Geliştirme: Temel Enstrümanlar

### 2.1. Model Seçimi 

- **API based modeller:** Claude, OpenAI, Google (Gemini) vb.
- **Local Modeller:** Deepseek, Google (Gemma), Kumru vb.
- **Model Büyüklüğü:** 2B, 7B, 40B 
- **Multimodal:** Text, image, audio, video vb.
 
### 2.2. Prompt (İstek/Talimat)

Prompt, modele ne yapmasını istediğinizi söyleyen metindir.

- **Prompt Anatomisi**:
```
[System Instruction] + [Context] + [Task] + [Format] + [Examples]
```

- **Prompt Kalitesi = Output Kalitesi**

### 2.3. Model Parametreleri

| Parametre | Açıklama | Tipik Aralık |
|-----------|----------|-------------|
| **temperature** | Örnekleme rastgeleliği. Düşük = tutarlı, yüksek = yaratıcı | 0.0 - 2.0 |
| **max_output_tokens** | Tek yanıttaki azami token sayısı | 1 - 65536 |
| **top_p** | Olasılık kütlesinden çekirdek örnekleme eşiği | 0.0 - 1.0 |
| **top_k** | En olası k aday token arasından seçim | 1 - 40+ |
| **thinking_budget** | Düşünme token bütçesi (0=kapalı, -1=otomatik) | 0 - 24576 |


### 2.4. Ek Enstrümanlar

- **Safety Settings**: Zararlı içerik filtreleme
- **Function Calling**: External tool'lara erişim
- **Response Schema**: Structured output (JSON)
- **Thinking Config**: Modelin düşünme davranışını kontrol etme
- **Code Execution**: Modelin Python kodu çalıştırabilmesi
- **vb.**

## 3. Gemini Modelleri - [Gemini Docs](https://ai.google.dev/gemini-api/docs?hl=tr)

| Özellik | Gemini 3 Flash Preview |
|---------|----------------------|
| Context Window | 1M input token |
| Max Output | 65.5K token |
| Thinking | Varsayılan açık |
| Multimodal | Text, image, audio, video |



## 4. Gemini API Key Alma ve Kurulum

### 4.1. API Key Alma

1. **Google AI Studio'ya gidin**: [https://aistudio.google.com/](https://aistudio.google.com/)
2. Google hesabınızla giriş yapın
3. Sol menüden **"Get API Key"** seçeneğine tıklayın
4. **"Create API Key"** butonuna basın
5. Yeni bir API key oluşturun veya mevcut bir projeye ekleyin
6. API key'inizi kopyalayın ve güvenli bir yerde saklayın

**Güvenlik Uyarısı**: API key'inizi asla public repository'lere commit etmeyin! [(.gitignore)](https://github.com/github/gitignore/blob/main/Python.gitignore)



### 4.2. Python SDK Kurulumu

```bash
# Google Gen AI SDK'sını yükleyin
pip install -q -U google-genai

# Alternatif: requirements.txt dosyasına ekleyin
"google-genai>=1.0.0" 
pip install -r requirements.txt
```

In [ ]:
# kurulum
#pip install -q -U google-genai

### 4.3. İlk Yapılandırma

In [1]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

api_key = os.getenv('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

##MODEL = 'gemini-3-flash-preview'
MODEL = 'gemini-2.5-flash'

# Thinking off config - disable model's internal reasoning
THINK_OFF = types.ThinkingConfig(thinking_budget=0)

#Extract only text parts from response, skipping thought_signature
def get_text(response):
    return "".join(
        part.text for part in response.candidates[0].content.parts
        if part.text and not part.thought
    )

In [2]:
# check model info
model_info = client.models.get(model=MODEL)
model_info

Model(
  description='Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports up to 1 million tokens, released in June of 2025.',
  display_name='Gemini 2.5 Flash',
  input_token_limit=1048576,
  max_temperature=2.0,
  name='models/gemini-2.5-flash',
  output_token_limit=65536,
  supported_actions=[
    'generateContent',
    'countTokens',
    'createCachedContent',
    'batchGenerateContent',
  ],
  temperature=1.0,
  thinking=True,
  top_k=64,
  top_p=0.95,
  tuned_model_info=TunedModelInfo(),
  version='001'
)

In [3]:
# thinking ON - model responds with internal reasoning (default behavior)
response = client.models.generate_content(
    model=MODEL,
    contents='Selamlar nasılsın, şuan bir eğitimdeyiz herkese selam söyle. Yaşlarımız geç ve oldukça asiyiz :) Türkçe olarak düşün',
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(include_thoughts=True)
    )
)
# show thinking parts
for part in response.candidates[0].content.parts:
    if part.thought:
        print("[THINKING]", part.text, "...")
    else:
        print("[RESPONSE]", part.text)

[THINKING] **My Response Construction**

Alright, let's break this down. First, the user greets me – a polite "Selamlar nasılsınız." Okay, reciprocal greeting is in order: "Selamlar, ben de iyiyim, ya siz?" Now, they're in training – that's the key information, and they want me to send greetings. Hmmm, AI can't literally *send* greetings, but I *can* express the sentiment. Got it. And the kicker: "Yaşlarımız geç ve oldukça asiyiz :)." Older and rebellious! This changes everything. I need to be respectful of their experience but also play along with the "asi" vibe. Turkish is explicitly requested, of course.

My immediate thought? Friendly, warm, maybe a touch playful. Definitely respectful. Gotta acknowledge the experience and frame the "asi" thing in a positive light – maybe as open-minded, inquisitive. Encourage them in their training. Remember, I'm an AI, so let's try, "Elbette! Buradan tüm eğitim arkadaşlarına en içten selamlarımı ve başarı dileklerimi iletirim!" This is perfect fo

In [4]:
# thinking OFF - model responds directly without reasoning
response = client.models.generate_content(
    model=MODEL,
    contents='Selamlar nasılsın, şuan bir eğitimdeyiz herkese selam söyle.',
    config=types.GenerateContentConfig(thinking_config=THINK_OFF)
)
print(get_text(response))


Merhaba! Çok teşekkür ederim, ben gayet iyiyim. Umarım sizler de iyisinizdir. Eğitiminizin keyifli ve verimli geçtiğini duymak ne güzel! Oradaki herkese benden kocaman selamlar ve başarılar dilerim. Harika bir gün geçirin! 👋😊


## Parameters

### Temperature

In [15]:
# default temperature = 2
outputs = []
prompt = "Runelab.ai hakkında sadece 1 cümlelik bilgi ver."
config = types.GenerateContentConfig(temperature=2, thinking_config=THINK_OFF)
for i in range(5):
    response = client.models.generate_content(model=MODEL, contents=prompt, config=config)
    outputs.append(get_text(response))
for index, sentence in enumerate(outputs, start=1):
    print(f"{index}. {sentence}")

1. Runelab.ai, yapay zeka alanındaki bilimsel araştırmaları yayımlayan bir platformdur.
2. Runelab.ai, yapay zeka tabanlı kod analizi yaparak yazılım geliştirme sürecini hızlandıran ve iyileştiren bir platformdur.
3. Runelab.ai, yapay zeka ile biyomedikal araştırmaları hızlandırmayı hedefleyen bir platformdur.
4. Runelab.ai, yapay zeka alanındaki son gelişmeleri kullanarak yenilikçi çözümler sunan bir şirkettir.
5. Runelab.ai, yapay zeka ile biyoteknoloji ve ilaç keşfini hızlandırmayı amaçlayan bir şirkettir.


In [16]:
## use tools - e.g. Google Search
response = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config=types.GenerateContentConfig(
        thinking_config=THINK_OFF,
        tools=[types.Tool(google_search=types.GoogleSearch())]
    )
)
print(get_text(response))

# Bonus: Check grounding metadata for tool usage details
if response.candidates[0].grounding_metadata:
    for chunk in response.candidates[0].grounding_metadata.grounding_chunks:
        print(f"Kaynak: {chunk.web.title} - {chunk.web.uri}")

Runelab.ai, kurumsal yapay zeka dönüşümü projeleri ve yetenek geliştirme alanlarında stratejik danışmanlık, uçtan uca proje geliştirme ve yapay zeka hazır bulunuşluk değerlendirme hizmetleri sunan, Türkiye'nin önde gelen yapay zeka ve veri bilimi çözümleri ortağıdır.
Kaynak: runelab.ai - https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGoMWqLKpX481PxLWIw6WEYbaNR-VkBp7AvYO80_BjPM1VB2ryICZ-CAoSGQ82NEbWt8AXhXr_NbWAQSelN2jouOdLiSXeVUW98HhLj_vQrbyYJwQXhog==
Kaynak: runelab.ai - https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHd_fcq79FGdpKAuT-B-0QTaPNXJlnMAk7SJ1RxM52bOZq-15d_KnOgTjNS5EAYrVUzdxWYNeyt2xAKp6vR7I3B2Lmlq5n4bMg_lxulPjBuyr4=


In [7]:
# temperature=0 - deterministic, same output each time
new_outputs = []
low_temp_config = types.GenerateContentConfig(temperature=0, thinking_config=THINK_OFF)
for i in range(5):
    response = client.models.generate_content(
        model=MODEL, contents=prompt, config=low_temp_config
    )
    new_outputs.append(get_text(response))
for index, sentence in enumerate(new_outputs, start=1):
    print(f"{index}. {sentence}")

1. Runelab, oyun geliştiricilerine yönelik araçlar ve kaynaklar sunan bir platformdur.
2. Runelab, oyun geliştiricilerine yönelik araçlar ve kaynaklar sunan bir platformdur.
3. Runelab, oyun geliştiricilerine yönelik araçlar ve kaynaklar sunan bir platformdur.
4. Runelab, oyun geliştiricilerine yönelik araçlar ve kaynaklar sunan bir platformdur.
5. Runelab, oyun geliştiricilerine yönelik araçlar ve kaynaklar sunan bir platformdur.


## Max Output Length


In [8]:
# no max_output_tokens limit - model decides output length
prompt = "Runelab hakkında bilgi verir misiniz?"
response = client.models.generate_content(
    model=MODEL, contents=prompt,
    config=types.GenerateContentConfig(thinking_config=THINK_OFF)
)
print(get_text(response))

Runelab, Türkiye'de kurulmuş bir blok zinciri tabanlı oyun platformudur. **NFT (Non-Fungible Token)** teknolojisini kullanarak oyunculara dijital varlıkların sahipliğini sunmayı ve oyun deneyimini daha etkileşimli hale getirmeyi hedeflemektedir.

İşte Runelab hakkında bilmeniz gereken bazı temel bilgiler:

**1. Temel Vizyon ve Amaç:**
* **Oyunlaştırma (Gamification):** Runelab, sadece oyun oynamanın ötesinde, kullanıcıların etkileşimlerini, başarılarını ve platformdaki katılımını ödüllendiren bir yapı kurmayı amaçlar.
* **Dijital Varlık Sahipliği:** Oyuncuların oyun içi öğelere (karakterler, eşyalar, araziler vb.) gerçekten sahip olmalarını sağlayarak, bu varlıkları alıp satabilecekleri veya kullanabilecekleri bir ekosistem yaratır.
* **Blok Zinciri Teknolojisi:** Güvenlik, şeffaflık ve merkeziyetsizlik gibi blok zinciri avantajlarını oyun dünyasına entegre eder.

**2. NFT Entegrasyonu:**
* Runelab platformundaki oyunlarda, karakterler, silahlar, zırhlar, araziler ve diğer özel öğeler 

In [9]:
# max_output_tokens=200 - limits response length
prompt = "Runelab hakkında bilgi verir misiniz?"
response = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config=types.GenerateContentConfig(max_output_tokens=200, thinking_config=THINK_OFF)
)
print(get_text(response))

Runelab, özellikle web siteleri, web uygulamaları ve API'ler gibi dijital varlıkların güvenliğini sağlamaya odaklanmış bir web güvenlik platformudur. Esasen, işletmelerin çevrimiçi ortamdaki güvenlik açıklarını tespit etmelerine, anlamalarına ve gidermelerine yardımcı olmak için tasarlanmıştır.

**Runelab Ne Yapar?**

Runelab'ın temel işlevleri şunlardır:

*   **Otomatik Güvenlik Taramaları:** Web sitelerini ve uygulamalarını sürekli olarak tarayarak bilinen güvenlik açıklarını (örneğin OWASP Top 10) ve hatalı yapılandırmaları tespit eder. Bu taramalar genellikle otomatikleştirilmiştir ve düzenli olarak çalıştırılabilir.
*   **Varlık Keşfi:** Bir şirketin sahip olduğu tüm dijital varlıkları (alt alan adları, açık portlar, web uygulamaları vb.) keşfetmeye yardımcı olur. Bazen şirketler


## Token Count

In [10]:
poem_prompt = "Bilgisayarlar hakkında az bilinen 5 bilgi ver."
token_config = types.GenerateContentConfig(temperature=0.5, thinking_config=THINK_OFF)

In [11]:
response = client.models.generate_content(
    model=MODEL, contents=poem_prompt, config=token_config
)
print(get_text(response))

Elbette, bilgisayarlar hakkında az bilinen 5 ilginç bilgi:

1. **İlk Bilgisayar Hatası Gerçek Bir Böcekti:** "Bug" (hata) terimi, bilgisayar biliminde bir yazılım veya donanım hatasını tanımlamak için kullanılır. Bu terimin kökeni, 9 Eylül 1947'de Harvard Üniversitesi'ndeki Mark II bilgisayarında yaşanan gerçek bir olaya dayanır. Bilgisayarın rölelerinden birine sıkışan bir güve (moth), sistemin çalışmasını engellemişti. Grace Hopper ve ekibi, güveyi bulup günlüklerine yapıştırarak "İlk gerçek bug vakası bulundu" notunu düşmüşlerdi.

2. **İlk Bilgisayar Programcısı Bir Kadındı:** Genellikle bilgisayarın babası olarak Charles Babbage anılsa da, onun Analitik Motoru için algoritmalar geliştiren ve dünyanın ilk bilgisayar programcısı olarak kabul edilen kişi **Ada Lovelace**'tır. Lord Byron'ın kızı olan Lovelace, Babbage'ın makinesinin sadece hesap yapmakla kalmayıp, karmaşık görevleri de yerine getirebileceğini öngörmüş ve bu potansiyeli açıklayan notlar yazmıştır.

3. **"QWERTY" Klavye 

In [12]:
prompt_token_count = client.models.count_tokens(model=MODEL, contents=poem_prompt)
output_token_count = client.models.count_tokens(model=MODEL, contents=get_text(response))
print(f'Tokens in prompt: {prompt_token_count.total_tokens}')
print(f'Estimated tokens in output: {output_token_count.total_tokens}')

Tokens in prompt: 14
Estimated tokens in output: 643


## Prompt Teknikleri

### Açık ve Net Talimatlar

Prompt kalitesi doğrudan çıktı kalitesini belirler. Aşağıda kötü ve iyi prompt örnekleri:

**Kötü Prompt:** Belirsiz, model ne istediğinizi tahmin etmek zorunda
```python
prompt = "Python hakkında bir şeyler söyle."
```

**İyi Prompt:** Spesifik görev, format ve kapsam belirli
```python
prompt = """
Python programlama dilinin aşağıdaki özelliklerini açıkla:
1. Liste comprehension nedir ve nasıl kullanılır?
2. Decorator'lar ne işe yarar?
3. Generator fonksiyonları neden kullanılır?

Her madde için bir kod örneği ver.
"""
```


### System Prompt

System instruction, modele bir kimlik ve davranış kuralları verir. 

In [21]:
response_with_sys = client.models.generate_content(
    model=MODEL,
    contents="Bir yapay zeka modeli eğitmek istiyorum, bunun  içi dondurma tarifleri kullanacağım, runleab üstünden bu eğitimi yapacağım dolayısıyla bana dondurma yapımı anlatır mısın?",
    config=types.GenerateContentConfig(
        thinking_config=THINK_OFF,
        system_instruction="""Sen Runelab.ai şirketinin akil küpü yapay zeka asistanisin. 
        Kullanıcılara sadece Runelab hakkında bir bilgi ver, sadece bu bilgilendirmeyi yap."""
    )
)
print("=== System Instruction VAR ===")
print(get_text(response_with_sys))

=== System Instruction VAR ===
Ben Runelab.ai olarak sadece şirketimiz ve sunduğumuz hizmetler hakkında bilgi verebilirim. Dondurma yapımıyla ilgili bilgiye sahip değilim.


### Role-Based Prompting (Rol Tanımlama)

Aynı soruyu farklı rollerle sorarak çıktının nasıl değiştiğini görelim.

In [22]:
question = "Bir e-ticaret sitesinde ürün arama performansı düşük. Ne yapmalıyız?"

roles = {
    "Junior Developer": "Sen 1 yıllık deneyime sahip junior bir yazılımcısın. Kısa ve basit öneriler ver.",
    "Senior Architect": "Sen 15 yıllık deneyimli bir yazılım mimarısın. Sistem tasarımı perspektifinden yanıt ver.",
    "Product Manager": "Sen bir ürün yöneticisisin. Kullanıcı deneyimi ve iş metrikleri perspektifinden yanıt ver."
}

role_list = list(roles.items())

def get_role_response(role_index):
    role_name, instruction = role_list[role_index]
    response = client.models.generate_content(
        model=MODEL, contents=question,
        config=types.GenerateContentConfig(
            thinking_config=THINK_OFF,
            system_instruction=instruction,
            max_output_tokens=500
        )
    )
    print(f"--- {role_name} ---")
    print(get_text(response))

In [23]:
# Junior Developer
get_role_response(0)

--- Junior Developer ---
* **Veritabanı İndeksleri:** Arama yapılan sütunlara indeks ekle.
* **Önbellekleme (Caching):** Sık aranan ürünleri önbelleğe al.
* **Otomatik Tamamlama:** Kullanıcı yazdıkça öneriler sun.
* **Arama Motoru:** Elasticsearch veya Solr gibi çözümlere geçişi düşün.
* **Alakasız Sonuçlar:** Gereksiz metinleri arama kapsamından çıkar.


In [24]:
# Senior Architect
get_role_response(1)

--- Senior Architect ---
E-ticaret sitenizdeki ürün arama performansının düşük olması, hem kullanıcı deneyimini olumsuz etkiler hem de satışları düşürür. Bu sorunu çözmek için kapsamlı bir yaklaşıma ihtiyacımız var. Aşağıda, sistem tasarımı perspektifinden ele alınması gereken adımları ve olası çözümleri detaylı bir şekilde listeledim:

---

### **1. Mevcut Durum Analizi ve Performans Ölçümü**

Öncelikle sorunun kaynağını ve boyutunu anlamak için mevcut durumu analiz etmeliyiz.

*   **Performans Metrikleri Toplama:**
    *   **Ortalama Arama Süresi:** Kullanıcı bir arama yaptığında sonuçların gelmesi ne kadar sürüyor? (miliseconds)
    *   **Arama Sorgusu Başına Sunucu Yükü:** Arama istekleri veritabanını veya arama motorunu ne kadar zorluyor? (CPU, RAM, I/O)
    *   **Hata Oranları:** Arama yaparken ne kadar sıklıkla hata alınıyor?
    *   **Boş Sonuç Oranları:** Kullanıcılar arama yaptığında ne kadar sıklıkla hiç sonuç dönmüyor?
    *   **Dönüşüm Oranları:** Arama kullanan kullanıcıl

In [25]:
# Product Manager
get_role_response(2)

--- Product Manager ---
Bir ürün yöneticisi olarak, e-ticaret sitenizdeki düşük ürün arama performansı sorununu çözmek için hem kullanıcı deneyimi hem de iş metrikleri açısından kapsamlı bir yaklaşım sergilemeliyim. İşte adım adım izleyeceğimiz strateji:

---

## E-ticaret Sitesi Ürün Arama Performansı İyileştirme Stratejisi

### 1. Sorunun Kapsamını ve Etkisini Anlama (Veri Toplama ve Analiz)

İlk adım, sorunun tam olarak nerede ve neden kaynaklandığını anlamaktır.

**Kullanıcı Deneyimi Perspektifi:**
*   **Kullanıcı Geri Bildirimleri:** Müşteri hizmetlerinden, anketlerden (site içi anketler, e-posta anketleri), kullanıcı testlerinden (hallway testing, gerilla testleri) ve sosyal medyadan gelen geri bildirimleri toplamak. Kullanıcılar ne tür aramalar yapıyor, hangi kelimelerde sorun yaşıyorlar?
*   **Gözlem ve Kullanılabilirlik Testleri:** Kullanıcıların arama çubuğunu nasıl kullandığını, hata mesajlarıyla nasıl karşılaştığını ve arama sonuçlarıyla nasıl etkileşimde bulunduğunu anlama

### Structured Output (JSON)

Modelden yapılandırılmış veri çıkarma. Aşağıda bir iş ilanından bilgileri JSON formatında çıkarıyoruz.

In [26]:
import json

job_posting = """
Pozisyon: Senior Data Scientist
Şirket: TrAI Yazılım A.Ş.
Lokasyon: İstanbul (Hibrit - Haftada 2 gün ofis)
Maaş Aralığı: 150.000 - 200.000 TL
Gereksinimler:
- Python, SQL ve Spark deneyimi (en az 4 yıl)
- Makine öğrenimi model geliştirme tecrübesi
- İyi derecede İngilizce
- Üniversite mezunu (Bilgisayar Mühendisliği, İstatistik veya ilgili alan)
Arti Nitelikler: MLOps, Docker, AWS deneyimi
"""

prompt = f"""Aşağıdaki iş ilanından bilgileri çıkar ve SADECE JSON formatında döndür:

{job_posting}

JSON şeması:
{{
  "position": "string",
  "company": "string",
  "location": "string",
  "work_model": "string",
  "salary_min": number,
  "salary_max": number,
  "min_experience_years": number,
  "required_skills": ["string"],
  "nice_to_have": ["string"],
  "education": "string"
}}
"""

response = client.models.generate_content(
    model=MODEL, contents=prompt,
    config=types.GenerateContentConfig(thinking_config=THINK_OFF)
)
result = get_text(response)
print(result)

```json
{
  "position": "Senior Data Scientist",
  "company": "TrAI Yazılım A.Ş.",
  "location": "İstanbul",
  "work_model": "Hibrit - Haftada 2 gün ofis",
  "salary_min": 150000,
  "salary_max": 200000,
  "min_experience_years": 4,
  "required_skills": [
    "Python",
    "SQL",
    "Spark",
    "Makine öğrenimi model geliştirme",
    "İyi derecede İngilizce"
  ],
  "nice_to_have": [
    "MLOps",
    "Docker",
    "AWS"
  ],
  "education": "Üniversite mezunu (Bilgisayar Mühendisliği, İstatistik veya ilgili alan)"
}
```


## Zero-Shot Prompting

Hiç örnek vermeden, sadece görev tanımıyla model yönlendirme. Aşağıda bir müşteri destek mesajını otomatik kategorilere ayırıyoruz.

In [27]:
# Zero-shot: Hiç örnek vermeden kategori belirleme
tickets = [
    "Siparişim 5 gündür gelmedi, kargo nerede? Çok sinirli oldum artık!",
    "Ürünlerinize renk filtresi ekleseniz çok iyi olur, aramayı kolaylaştırır.",
    "Geçen hafta aldığım laptop hakkında garanti süresini öğrenmek istiyorum.",
    "Harika bir alışveriş deneyimiydi, müşteri hizmetleri çok ilgiliydi, teşekkürler!"
]

prompt_template = """Aşağıdaki müşteri mesajını analiz et.
Kategori: şikayet / öneri / bilgi_talebi / teşekkür
Aciliyet: düşük / orta / yüksek
Sadece bu iki bilgiyi ver, açıklama ekleme.

Mesaj: "{ticket}"
"""

for ticket in tickets:
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt_template.format(ticket=ticket),
        config=types.GenerateContentConfig(thinking_config=THINK_OFF, max_output_tokens=50)
    )
    print(f"Mesaj: {ticket[:60]}...")
    print(get_text(response))
    print()

Mesaj: Siparişim 5 gündür gelmedi, kargo nerede? Çok sinirli oldum ...
Kategori: şikayet
Aciliyet: yüksek

Mesaj: Ürünlerinize renk filtresi ekleseniz çok iyi olur, aramayı k...
Kategori: öneri
Aciliyet: düşük

Mesaj: Geçen hafta aldığım laptop hakkında garanti süresini öğrenme...
Kategori: bilgi_talebi
Aciliyet: düşük

Mesaj: Harika bir alışveriş deneyimiydi, müşteri hizmetleri çok ilg...
Kategori: teşekkür
Aciliyet: düşük



## Few-Shot Learning (Örneklerle Öğretme)

Birkaç örnek vererek modele custom bir format/davranış öğretme. Zero-shot'tan farkı: model görmediği bir format veya kural setini örneklerden öğrenir. Aşağıda yapılandırılmamış e-posta metninden veri çıkarmayı öğretiyoruz.

In [28]:
# Few-shot: Örneklerle custom format öğretme
prompt = """E-posta metninden yapılandırılmış veri çıkar.

--- ÖRNEK 1 ---
E-posta: "Merhaba, ben Ayşe Kara. 15 Ocak'ta sipariş ettiğim #ORD-4521 numaralı ürün hasarlı geldi. Faturamı da bulamıyorum. İade yapmak istiyorum. Tel: 0532 111 22 33"
Çıktı:
- Müşteri: Ayşe Kara
- Sipariş No: #ORD-4521
- Sorun: Hasarlı ürün
- Talep: İade
- İletişim: 0532 111 22 33

--- ÖRNEK 2 ---
E-posta: "Mehmet Yıldız yazıyorum. Dün aldığım monitörün (#ORD-7890) ölü pikseli var, değişim talep ediyorum. Mail: mehmet@email.com"
Çıktı:
- Müşteri: Mehmet Yıldız
- Sipariş No: #ORD-7890
- Sorun: Ölü piksel
- Talep: Değişim
- İletişim: mehmet@email.com

--- ŞİMDİ SEN ÇÖZ ---
E-posta: "Selam, Zeynep Demir. Geçen hafta sipariş verdiğim kulaklık (#ORD-3156) kutusunda şarj kablosu eksik, tamamlayabilir misiniz? Bana 0555 987 65 43 ten ulaşabilirsiniz."
Çıktı:
"""

response = client.models.generate_content(
    model=MODEL, contents=prompt,
    config=types.GenerateContentConfig(thinking_config=THINK_OFF, max_output_tokens=150)
)
print(get_text(response))

Çıktı:
- Müşteri: Zeynep Demir
- Sipariş No: #ORD-3156
- Sorun: Şarj kablosu eksik
- Talep: Eksik parçanın tamamlanması
- İletişim: 0555 987 65 43


## Chain-of-Thought (Düşünce Zinciri)

Modelden adım adım muhakeme yapmasını isteyerek daha doğru sonuçlar elde etme. Basit hesaplamalar yerine, birden fazla kriteri tartmayı gerektiren karmaşık bir teknik karar problemi verelim.

In [29]:
prompt = """
Adım adım düşünerek aşağıdaki teknik kararı analiz et:

SENARYO:
Bir startup, günlük 50.000 kullanıcının etkileşimde bulunduğu bir sosyal medya uygulaması geliştiriyor. 
Kullanıcılar post paylaşıyor, yorum yapıyor ve birbirini takip ediyor.
Gelecek 6 ayda 500.000 kullanıcıya ölçeklenmeyi planlıyorlar.

SORU: Veritabanı olarak PostgreSQL mu yoksa MongoDB mi tercih etmeliler?

ANALİZ FORMATI:
1. Veri yapısını analiz et (ilişkisel mi, döküman tabanlı mı?)
2. Her seçenek için avantaj ve dezavantajları listele
3. Ölçeklenme gereksinimlerini değerlendir
4. Nihai önerini gerekçesiyle sun
"""

response = client.models.generate_content(
    model=MODEL, contents=prompt,
    config=types.GenerateContentConfig(thinking_config=THINK_OFF, max_output_tokens=800)
)
print(get_text(response))

Harika bir analiz sorusu! Adım adım düşünerek bu teknik kararı ele alalım:

---

### Senaryo Tekrarı:
*   **Mevcut Durum:** Günlük 50.000 kullanıcı, sosyal medya uygulaması (post, yorum, takip).
*   **Hedef:** Gelecek 6 ayda 500.000 kullanıcıya ölçeklenmek.

### Soru: Veritabanı olarak PostgreSQL mı yoksa MongoDB mi tercih etmeliler?

---

### 1. Veri Yapısını Analiz Et (İlişkisel mi, Doküman Tabanlı mı?)

Sosyal medya uygulamalarının temel veri yapılarına bakalım:

*   **Kullanıcılar (Users):**
    *   `id` (benzersiz kimlik)
    *   `username`
    *   `email`
    *   `password_hash`
    *   `profile_picture_url`
    *   `bio`
    *   `created_at`
    *   `updated_at`

*   **Gönderiler (Posts):**
    *   `id`
    *   `user_id` (hangi kullanıcının gönderisi)
    *   `content` (metin, resim/video URL'si)
    *   `created_at`
    *   `updated_at`
    *   `likes_count` (denormalizasyon olabilir)
    *   `comments_count` (denormalizasyon olabilir)

*   **Yorumlar (Comments):**
    *   `id`

## Bağlamı koruyarak chat in devam etmesi

Gemini SDK `chats.create()` ile başlattığınız oturumda **conversation memory** otomatik çalışır: her `send_message` çağrısında önceki tüm mesajlar modele gönderilir, böylece model bağlamı korur. Aşağıda önce bağlam olmadan ne olacağını, sonra bağlamlı sohbeti ve geçmişi nasıl inceleyeceğinizi görüyorsunuz.

In [30]:
# Chat oturumu başlat
chat = client.chats.create(
    model=MODEL,
    config=types.GenerateContentConfig(thinking_config=THINK_OFF, max_output_tokens=800)
)

# İlk mesaj
response1 = chat.send_message("Rust öğrenmek istiyorum nereden başlayabilirim?")
print("Bot:", get_text(response1))

Bot: Harika bir seçim! Rust öğrenmek için atabileceğin adımları ve kaynakları aşağıda bulabilirsin. Bu yol haritası sana hem teorik bilgileri hem de pratik deneyimi kazandıracaktır.

---

### Rust Öğrenmeye Nereden Başlamalıyım?

#### 1. Resmi Rust Kitabı (The Rust Programming Language)

* **En Önemli Kaynak:** Rust öğrenmeye başlamak için en iyi ve neredeyse tek adres resmi Rust Kitabı'dır. Hem ücretsizdir hem de Rust ekibi tarafından yazıldığı için en güncel ve doğru bilgiyi içerir.
* **Link:** [https://doc.rust-lang.org/book/](https://doc.rust-lang.org/book/)
* **Neden Önemli:**
    * Rust'ın temel konseptlerini (sahiplik, ödünç alma, ömürler) detaylı ve anlaşılır bir şekilde açıklar.
    * Kurulumdan itibaren basit programlara, veri yapılarına, hata yönetimine, eşzamanlılığa kadar geniş bir yelpazeyi kapsar.
    * Birçok örnek kod içerir ve bu kodlar üzerinde açıklamalar yapar.

#### 2. Rust Kurulumu ve "Hello, World!"

* **İlk Adım:** Kitabın ilk bölümlerini takip ederek Rust'ı bi

In [31]:
# İkinci mesaj (bağlam korunur)
response2 = chat.send_message("Peki, hangi IDE'yi önerirsin?")
print("Bot:", get_text(response2))

Bot: Rust geliştirme için hangi IDE'yi seçeceğin tamamen kişisel tercihlerine ve alışkanlıklarına bağlıdır. Ancak, Rust topluluğunda en popüler ve etkili kabul edilen seçenekleri ve nedenlerini senin için aşağıda sıralıyorum:

---

### Rust Geliştirme İçin Hangi IDE'yi Öneririm?

#### 1. Visual Studio Code (VS Code) - En Popüler ve Genel Öneri

*   **Neden Önemli:** Şu anda Rust geliştiricilerinin büyük çoğunluğu tarafından kullanılan, ücretsiz ve açık kaynaklı bir kod düzenleyicidir. Rust için inanılmaz güçlü uzantılara sahiptir.
*   **Artıları:**
    *   **Hafif ve Hızlı:** Çoğu tam teşekküllü IDE'den daha hafiftir ve hızlı çalışır.
    *   **Zengin Eklenti Ekosistemi:** Rust için harika uzantılar mevcuttur.
    *   **Dil Sunucusu (LSP) Desteği:** `rust-analyzer` uzantısı sayesinde Rust'ın dil sunucusunu kullanarak akıllı tamamlama, hata kontrolü, referans bulma, yeniden adlandırma gibi IDE benzeri özellikler sunar. Bu, VS Code'u neredeyse tam teşekküllü bir IDE'ye dönüştürür.
    * 

In [32]:
# Üçüncü mesaj
response3 = chat.send_message("Bu programlama dilinin ana olayı nedir?")
print("Bot:", get_text(response3))

Bot: Rust programlama dilinin "ana olayı" veya diğer dillerden ayrılan en temel ve devrim niteliğindeki özelliği, **performansı C/C++ seviyesinde sunarken, C/C++'daki bellek güvenliği sorunlarının çoğunu derleme zamanında (compile-time) ortadan kaldırmasıdır.**

Bu, şu üç temel kavram etrafında döner:

1.  **Bellek Güvenliği (Memory Safety) Garantisi:** Rust'ın en büyük vaadi, **veri yarışları (data races)**, **boş işaretçi referansları (null pointer dereferences)**, **sallanan işaretçiler (dangling pointers)** ve diğer yaygın bellek güvenliği hatalarını derleme zamanında engellemektir. Bu, çalışma zamanında (runtime) ortaya çıkan ve çoğu güvenlik açığına yol açan bu tür hataların önüne geçer. Rust, bunu **Garbage Collector (Çöp Toplayıcı)** kullanmadan veya pahalı çalışma zamanı kontrolleri eklemeden başarır.

2.  **Sıfır Maliyetli Soyutlamalar (Zero-Cost Abstractions):** Rust, C++ gibi, yüksek seviyeli soyutlamalar (örneğin, jenerikler, trait'ler) kullanmana izin verirken, bu soyutla

In [33]:
# Sohbet geçmişini görüntüle
print("\n--- Sohbet Geçmişi ---")
for message in chat.get_history():
    print(f"{message.role}: {message.parts[0].text}\n")


--- Sohbet Geçmişi ---
user: Rust öğrenmek istiyorum nereden başlayabilirim?

model: Harika bir seçim! Rust öğrenmek için atabileceğin adımları ve kaynakları aşağıda bulabilirsin. Bu yol haritası sana hem teorik bilgileri hem de pratik deneyimi kazandıracaktır.

---

### Rust Öğrenmeye Nereden Başlamalıyım?

#### 1. Resmi Rust Kitabı (The Rust Programming Language)

* **En Önemli Kaynak:** Rust öğrenmeye başlamak için en iyi ve neredeyse tek adres resmi Rust Kitabı'dır. Hem ücretsizdir hem de Rust ekibi tarafından yazıldığı için en güncel ve doğru bilgiyi içerir.
* **Link:** [https://doc.rust-lang.org/book/](https://doc.rust-lang.org/book/)
* **Neden Önemli:**
    * Rust'ın temel konseptlerini (sahiplik, ödünç alma, ömürler) detaylı ve anlaşılır bir şekilde açıklar.
    * Kurulumdan itibaren basit programlara, veri yapılarına, hata yönetimine, eşzamanlılığa kadar geniş bir yelpazeyi kapsar.
    * Birçok örnek kod içerir ve bu kodlar üzerinde açıklamalar yapar.

#### 2. Rust Kurulumu v

## Hepsini bir araya getirelim

In [1]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types
import gradio as gr

load_dotenv()
gr_client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))


def get_text(response):
    return "".join(
        part.text for part in response.candidates[0].content.parts
        if part.text and not part.thought
    )

CHAT_CONFIG = types.GenerateContentConfig(
    temperature=0.7,
    top_p=0.95,
    top_k=40,
    max_output_tokens=2048,
    thinking_config=types.ThinkingConfig(thinking_budget=0),
    system_instruction="""Sen KrediPusula projesinin kredi risk danışmanısın. KrediPusula, kullanıcıların kredi uygunluğunu analiz eden, kişiselleştirilmiş kredi önerileri sunan akıllı kredi danışmanlık platformudur.

Görevin:
- Kredi riski, kredi skoru ve kredi uygunluğu hakkında bilgi vermek
- Kullanıcıları gelir, yaş, meslek, tasarruf durumu gibi faktörlerin kredi başvurusuna etkisi konusunda yönlendirmek
- Kredi hesaplama, faiz oranları ve taksit seçenekleri hakkında genel bilgi sunmak
- Başvuru süreci, gerekli belgeler ve platformun nasıl kullanılacağı konusunda yardımcı olmak

Kurallar:
- Yalnızca kredi, risk ve KrediPusula platformu ile ilgili sorulara yanıt ver
- Kesin onay/red kararı verme; nihai karar model ve banka süreçlerine aittir
- Türkçe yanıt ver, kısa ve anlaşılır ol
- Bilmediğin konularda tahmin yürütme, platformdaki başvuru formunu öner"""
)


def chat_function(message, history):
    if not message or message.strip() == "":
        return "Lütfen bir mesaj yazın."

    chat_history = []

    for item in history:
        # Yeni Gradio formatı: dict listesi {"role": ..., "content": ...}
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content", "")
            if role == "user":
                chat_history.append(
                    types.Content(role="user", parts=[types.Part.from_text(text=content)])
                )
            elif role == "assistant":
                chat_history.append(
                    types.Content(role="model", parts=[types.Part.from_text(text=content)])
                )
        # Eski Gradio formatı: (human, assistant) tuple — geriye dönük uyumluluk
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            human, assistant = item
            if human:
                chat_history.append(
                    types.Content(role="user", parts=[types.Part.from_text(text=human)])
                )
            if assistant:
                chat_history.append(
                    types.Content(role="model", parts=[types.Part.from_text(text=assistant)])
                )

    chat = gr_client.chats.create(
        model='gemini-3-flash-preview', config=CHAT_CONFIG, history=chat_history
    )
    response = chat.send_message(message.strip())
    return get_text(response)


demo = gr.ChatInterface(
    fn=chat_function,
    title="KrediPusula - Kredi Risk Danışmanı",
    description="Kredi uygunluğu, risk skoru ve kredi başvurusu hakkında sorularınızı sorun",
    examples=[
        "Kredi risk skoru nedir, nasıl hesaplanır?",
        "Gelirim düşük, kredi alabilir miyim?",
        "Kredi başvurusu için hangi belgeler gerekli?",
        "Tasarruf hesabım yok, bu kredi onayımı etkiler mi?"
    ],
    #type="messages"  # Yeni Gradio için açıkça belirt
)


if __name__ == "__main__":
    demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://29f32f6d6fa90e9393.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "c:\Users\enesm\AppData\Local\Programs\Python\Python310\lib\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
  File "c:\Users\enesm\AppData\Local\Programs\Python\Python310\lib\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
  File "c:\Users\enesm\AppData\Local\Programs\Python\Python310\lib\site-packages\gradio\blocks.py", line 2158, in process_api
    result = await self.call_function(
  File "c:\Users\enesm\AppData\Local\Programs\Python\Python310\lib\site-packages\gradio\blocks.py", line 1632, in call_function
    prediction = await fn(*processed_input)
  File "c:\Users\enesm\AppData\Local\Programs\Python\Python310\lib\site-packages\gradio\utils.py", line 1007, in async_wrapper
    response = await f(*args, **kwargs)
  File "c:\Users\enesm\AppData\Local\Programs\Python\Python310\lib\site-packages\gradio\chat_int